# 01 — Exploration des données de volatilité

Exploration rapide des sorties du pipeline : prix, rendements, volatilités réalisées, prime de risque de volatilité et prédictions walk-forward.

Prérequis : avoir lancé `python scripts/run_download.py` puis `python scripts/run_features.py` (et `run_train.py` pour la dernière section).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.utils.io import load_config, load_dataframe

config = load_config()
processed_dir = Path(config["data"]["processed_dir"])

## Prix et rendements

In [ ]:
prices = load_dataframe(processed_dir / "prices.csv")
returns = load_dataframe(processed_dir / "log_returns.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
prices[["SPY", "QQQ"]].plot(ax=axes[0], title="Prix ajustés")
prices[["^VIX", "^VIX3M"]].plot(ax=axes[1], title="VIX et VIX3M (vol implicite)")
plt.tight_layout()

## Volatilité réalisée vs implicite et term structure

In [ ]:
features = load_dataframe(processed_dir / "features.csv")

ax = features[["rv_20", "gk_20", "implied_vol"]].plot(figsize=(12, 4), alpha=0.8)
ax.set_title("Vol réalisée 20j (close-to-close et Garman-Klass) vs vol implicite — SPY")
ax.set_ylabel("Vol annualisée")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
features["implied_minus_rv"].plot(ax=axes[0], title="Prime de risque de volatilité (implicite - réalisée)")
axes[0].axhline(0, color="k", lw=0.5)
features["vix_term_structure"].plot(ax=axes[1], title="Term structure VIX/VIX3M (>1 = backwardation)")
axes[1].axhline(1, color="k", lw=0.5)
plt.tight_layout()

## Target et corrélations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3))
features["future_rv"].hist(bins=80, ax=axes[0])
axes[0].set_title("Vol réalisée future 5j (niveau)")
features["target"].hist(bins=80, ax=axes[1])
axes[1].set_title(f"Target transformée ({config['features']['target_type']})")
plt.tight_layout()
plt.show()

features.corr()["target"].drop(["target", "future_rv"]).sort_values(ascending=False)

## Prédictions walk-forward (après `run_train.py`)

In [ ]:
predictions_path = processed_dir / "predictions_rv.csv"
if (Path.cwd().parent / predictions_path).exists():
    preds = load_dataframe(predictions_path)
    cols = ["ensemble", "random_forest", "har", "y_true_rv"]
    ax = preds[cols].plot(figsize=(13, 4), alpha=0.8, title="Prédictions vs réalité (walk-forward, espace RV)")
    ax.set_ylabel("Vol annualisée")

    band = preds[["gb_q10", "gb_q90"]].tail(500)
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.fill_between(band.index, band["gb_q10"], band["gb_q90"], alpha=0.3, label="bande q10-q90")
    preds["y_true_rv"].tail(500).plot(ax=ax, color="k", lw=0.8, label="RV future réalisée")
    ax.legend(); ax.set_title("Incertitude de prédiction (quantile GB)")
else:
    print("Lancez d'abord: python scripts/run_train.py")